# Detector comparison on a GPU runtime (YOLOv8n · YOLOv8s · RT-DETR-l × base / augmented data)

Runtime → Change runtime type → **T4 GPU** (or better).
Every run uses the same tiles, hyper-parameters, epochs and test-year evaluation (`src/b5_eval.py`):
point FROC on 578 test frames (2010 · 2018), Pd at 0.25–8 false alarms per frame, frame-bootstrap 95% CI,
full-frame (1024 px) and sliced (320 px tiles) inference.

Rough time on a T4: data + tiles ≈ 10 min, YOLOv8n/s ≈ 5–15 min per run, RT-DETR-l ≈ 40–60 min per run, evaluation ≈ 20 min.

In [ ]:
BRANCH = "main"
!git clone -q -b $BRANCH https://github.com/jongchan-joseph-lee/sss-mine-detection-benchmark.git
%cd sss-mine-detection-benchmark/src
!pip -q install "ultralytics>=8.3" scipy scikit-learn

## 1. Data and tiles (official year split: train 2015 · 2017 · 2021, test 2010 · 2018)

In [ ]:
!bash download_data.sh > /dev/null
!python prep_tiles.py      # base tiles, val split, 1024-px test frames
!python prep_aug.py        # scale-down + shadow-aware copy-paste + speckle/JPEG (training scenes only)

## 2. Training grid

Same settings as `train_yolo.py` (320-px tiles, batch 16, flips, brightness only); only the model and data differ.
Set `SEEDS = [0, 1, 2]` to measure run-to-run variance if time allows.

In [ ]:
EPOCHS = 60
SEEDS = [0]
MODELS = {"n": "yolov8n.pt", "s": "yolov8s.pt", "rtdetr": "rtdetr-l.pt"}
DATA = {"base": "tiles/milco_tiles.yaml", "aug": "tiles/milco_tiles_aug.yaml"}
runs = {}
for mk, w in MODELS.items():
    for dk, y in DATA.items():
        for sd in SEEDS:
            name = f"{mk}_{dk}{EPOCHS}_s{sd}"
            !python train_exp.py --data {y} --model {w} --epochs {EPOCHS} --name {name} --device 0 --workers 4 --seed {sd}
            runs[name] = f"runs/detect/runs/exp/{name}/weights/best.pt"
runs

## 3. Test-year evaluation (reference = the CPU-trained YOLOv8n in the repository)

In [ ]:
import os
args = " ".join(["ref=runs/detect/runs/yolo_tiles/weights/best.pt"] + [f"{k}={v}" for k, v in runs.items() if os.path.exists(v)])
!B5_DEVICE=0 python b5_eval.py {args}

In [ ]:
import json, pandas as pd
r = json.load(open("b5_eval.json"))
rows = []
for k, v in r.items():
    g = v["fa_grid"]
    i1, i4 = g.index(1), g.index(4)
    rows.append(dict(run=k,
                     pd_at_1=v["pd"][i1], ci_1=v["ci95"][i1],
                     pd_at_4=v["pd"][i4], ci_4=v["ci95"][i4],
                     pd_30_50px_at_4=v["pd_by_size_at_4fa"]["30-50"]["pd"],
                     diff_vs_ref_at_1=v.get("diff_vs_ref_full", {}).get("mean", [None]*len(g))[i1],
                     diff_ci_at_1=v.get("diff_vs_ref_full", {}).get("ci95", [None]*len(g))[i1]))
pd.DataFrame(rows).set_index("run").round(3)

## 4. Save results (download, then copy into `src/` of the repository)

In [ ]:
!zip -q -r b5_colab_results.zip b5_eval.json b5_dets runs/detect/runs/exp/*/weights/best.pt runs/detect/runs/exp/*/results.csv
from google.colab import files
files.download("b5_colab_results.zip")